# vLLM vs Hugging Face — RunPod-ready setup

**Before running anything:** make sure your RunPod pod has a GPU attached. Recommended templates:
- *RunPod PyTorch 2.4* (CUDA 12.4) — works with vLLM 0.6.x
- *RunPod PyTorch 2.5* (CUDA 12.4) — works with vLLM 0.7.x / 0.8.x
- *RunPod PyTorch 2.6* (CUDA 12.6) — works with recent vLLM

The single most common error (`libcudart.so.XX: cannot open shared object file`) means vLLM's compiled binaries expect a different CUDA runtime than the one your container ships with. We'll detect that first and install a matching vLLM.

## Step 1 — Inspect what the container already has

Don't install anything yet. First find out which PyTorch + CUDA combo is preinstalled, because we'll match vLLM to it.

In [ ]:
import torch, sys, subprocess
print('Python    :', sys.version.split()[0])
print('PyTorch   :', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('GPU avail :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name  :', torch.cuda.get_device_name(0))
    print('VRAM (GB) :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print()
print('nvidia-smi driver / CUDA runtime:')
print(subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())

## Step 2 — Install vLLM that matches

**Critical rule:** do *not* `pip install torch` yourself. Either let vLLM bring its own torch, or pick a vLLM wheel built for the torch you already have. We do the latter — safer on RunPod because the container's torch is paired with the right CUDA libraries.

If Step 1 shows CUDA build `12.1`, install vLLM with `--extra-index-url` pointing at the cu121 wheels; for 12.4 use cu124, etc. The line below installs the latest vLLM compatible with whatever torch is present (pip will resolve).

In [ ]:
# Install vLLM WITHOUT touching torch (--no-deps for torch would be risky;
# instead we use the official vLLM install which respects existing torch).
!pip install -q vllm transformers accelerate sentencepiece matplotlib pandas

### If Step 2 *upgraded* torch on you

Run the cell below to see what happened. If `torch.__version__` changed compared to Step 1, you need to **restart the kernel** before importing vLLM, otherwise Python keeps the old torch in memory and you get `libcudart.so` errors.

**To restart in Jupyter:** Kernel → Restart Kernel. Then skip the install cell and continue from Step 3.

In [ ]:
import torch
print('PyTorch now:', torch.__version__, '| CUDA:', torch.version.cuda)

## Step 3 — Confirm vLLM imports cleanly

This is the moment of truth. If it fails with `libcudart.so.XX`, the torch and vLLM CUDA versions disagree. See the troubleshooting box at the end of this notebook.

In [ ]:
import os, torch, vllm
from vllm import LLM, SamplingParams
os.environ.setdefault('VLLM_USE_DEEP_GEMM', '0')  # disable FP8 DeepGEMM if not installed
print('vLLM   :', vllm.__version__)
print('PyTorch:', torch.__version__)
print('CUDA OK:', torch.cuda.is_available())

## Step 4 — Hugging Face baseline

Load Qwen1.5-1.8B with the standard transformers pipeline. This is the reference we'll race vLLM against.

Key choices and *why*:
- `trust_remote_code=True` — lets the Hub repo run its own tokenizer/model code (Qwen needs it).
- `pad_token = eos_token` — base Qwen has no pad token; we borrow EOS so batching works.
- `padding_side='left'` — for generation, the real prompt must sit flush at the right edge where the model predicts from. Right-padding would have it generating from `<pad>` tokens.
- `torch_dtype=torch.float16` — halves memory, negligible quality loss for inference.
- `device_map='auto'` — accelerate places layers on the GPU automatically.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

MODEL_NAME = 'Qwen/Qwen1.5-1.8B'

tokenizer_hf = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer_hf.pad_token is None:
    tokenizer_hf.pad_token = tokenizer_hf.eos_token
tokenizer_hf.padding_side = 'left'

model_hf = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)

pipe_hf = pipeline(
    'text-generation',
    model=model_hf,
    tokenizer=tokenizer_hf,
    max_new_tokens=10,
    batch_size=4,
    do_sample=False,   # greedy, so it matches vLLM and stays deterministic
)
print('HF pipeline ready.')

### Quick HF sanity check

In [ ]:
SYSTEM_PROMPT = '''Task: Determine if code needs migration from Python to C++.
Rules:
- If the code is Python: answer "Yes"
- If the code is C++ or other language: answer "No"
- Answer with only one word: "Yes" or "No"

Examples:
Code: def func(): pass
Answer: Yes

Code: #include <iostream>
Answer: No
'''

python_snippet = '''def add(a, b):
    return a + b'''

cpp_snippet = '''#include <iostream>
int add(int a, int b) { return a + b; }'''

js_snippet = '''function add(a, b) { return a + b; }'''

SNIPPETS = [python_snippet, cpp_snippet, js_snippet]

def make_prompt(code):
    return f"{SYSTEM_PROMPT}\nCode: {code}\nAnswer:"

for s in SNIPPETS:
    out = pipe_hf(make_prompt(s), return_full_text=False)[0]['generated_text']
    print(repr(out.strip().split('\n')[0]))

## Step 5 — vLLM engine

Same model, different engine. Tuning knobs that matter:
- `gpu_memory_utilization=0.85` — share of VRAM vLLM may claim for its KV cache (bigger = more concurrent requests).
- `max_model_len=1024` — caps prompt + output length. Qwen's default is 32k, which forces a huge KV cache reservation and can fail to start on small GPUs. Lower it when your prompts are short.
- `dtype='half'` — vLLM's name for float16.
- `enforce_eager=True` — skips torch.compile graph capture. Slightly slower per token but starts ~30s faster and avoids most compile-time errors. Drop this once you know everything else works.

In [ ]:
from vllm import LLM, SamplingParams

llm_vllm = LLM(
    model=MODEL_NAME,
    trust_remote_code=True,
    dtype='half',
    gpu_memory_utilization=0.85,
    max_model_len=1024,
    enforce_eager=True,
)

sampling = SamplingParams(
    max_tokens=10,
    temperature=0.0,
    stop=['\n', 'Code:', '###', 'Task:', 'Rules:'],
)
print('vLLM engine ready.')

### Quick vLLM sanity check

In [ ]:
prompts = [make_prompt(s) for s in SNIPPETS]
outs = llm_vllm.generate(prompts, sampling)
for o in outs:
    print(repr(o.outputs[0].text.strip()))

## Step 6 — Throughput benchmark

Same prompts, scaled up. The point isn't classification quality — it's measuring how each engine handles batch size. vLLM's continuous batching + paged KV cache should pull far ahead as the batch grows.

In [ ]:
import time, math
import pandas as pd
import matplotlib.pyplot as plt

def build_prompts(n):
    unit = [make_prompt(s) for s in SNIPPETS]
    return (unit * math.ceil(n / len(unit)))[:n]

def gpu_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def bench_hf(prompts):
    pipe_hf(prompts[:2], return_full_text=False)  # warmup
    gpu_sync(); t0 = time.perf_counter()
    outs = pipe_hf(prompts, return_full_text=False, batch_size=4)
    gpu_sync(); elapsed = time.perf_counter() - t0
    toks = sum(len(tokenizer_hf(o[0]['generated_text'], add_special_tokens=False).input_ids) for o in outs)
    return elapsed, len(prompts) / elapsed, toks / elapsed

def bench_vllm(prompts):
    llm_vllm.generate(prompts[:2], sampling)  # warmup
    gpu_sync(); t0 = time.perf_counter()
    outs = llm_vllm.generate(prompts, sampling)
    gpu_sync(); elapsed = time.perf_counter() - t0
    toks = sum(len(tokenizer_hf(o.outputs[0].text, add_special_tokens=False).input_ids) for o in outs)
    return elapsed, len(prompts) / elapsed, toks / elapsed

rows = []
for n in [3, 30, 300]:
    prompts = build_prompts(n)
    e, sps, tps = bench_hf(prompts)
    rows.append(dict(engine='HF', n=n, elapsed_s=e, samples_per_s=sps, tokens_per_s=tps))
    e, sps, tps = bench_vllm(prompts)
    rows.append(dict(engine='vLLM', n=n, elapsed_s=e, samples_per_s=sps, tokens_per_s=tps))

df = pd.DataFrame(rows)
print(df.to_string(index=False))

plt.figure(figsize=(7, 4))
for eng in df['engine'].unique():
    sub = df[df['engine'] == eng]
    plt.plot(sub['n'], sub['samples_per_s'], marker='o', label=eng)
plt.xscale('log'); plt.xlabel('Number of prompts'); plt.ylabel('Samples / sec')
plt.title('Throughput: HF vs vLLM'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('benchmark.png', dpi=150); plt.show()

## Troubleshooting: RunPod-specific errors

**`ImportError: libcudart.so.12 (or .so.13): cannot open shared object file`**

vLLM's binaries expect a CUDA runtime that isn't in this container. Two fixes:

1. *Easiest:* pick a newer RunPod template (PyTorch 2.5+ on CUDA 12.4 or 12.6) and start fresh.
2. *In-place:* pin a vLLM version that matches your torch. Example for a container with torch 2.1 / cu121:
   ```bash
   pip install -q vllm==0.6.3.post1
   ```
   Then **restart the kernel**.

**`OutOfMemoryError` during `LLM(...)` init**

Lower `gpu_memory_utilization` (try 0.7), lower `max_model_len`, or pick a smaller model. The KV cache reservation is what usually breaks this — `max_model_len=1024` instead of 32768 is a huge win on small GPUs.

**`RuntimeError: CUDA error: no kernel image is available for execution on the device`**

vLLM doesn't have a kernel for your GPU's compute capability. Common on older T4 / K80 RunPod offerings. Use an A4000, A5000, A100, or any newer Ampere/Hopper card.

**Hang at `Capturing CUDA graph shapes...`**

Set `enforce_eager=True` in the `LLM(...)` constructor (we already did above). torch.compile graph capture is the slow part of startup and the most failure-prone.